# ITCC508 Lab Exercise PT-F1 — FAISS Vector Store and Retrieval

**Mark Emmanuel P. Sagarino**
College of Computer Studies, Jose Rizal University

---

### What this notebook is

This is the provided `Faiss_Vector_Store_and_Retrieval` notebook, restructured for PT-F1. The two
original code cells have been split into one cell per logical function, each preceded by a text
cell explaining what that section does and why it is there. The original behaviour is preserved:
the same chunking algorithm, the same Excel chunk-inspection export, the same FAISS index built on
normalised vectors, and the same persistence files.

Two things have changed, both required by the exercise:

1. **The dataset.** The document indexed here is `Sagarino_PT-F1_Corpus.pdf`, a consolidated reference of
   the four Philippine statutes and the agency guidance that govern remedies for a defective
   consumer purchase. This is the corpus for my RAG application.
2. **The language model.** `google/flan-t5-base` has been replaced by the model selected for
   `openai/gpt-oss-20b`, served through the Groq API. Component II has been rewritten accordingly,
   with a grounding prompt and guardrails.

### Pipeline

| Phase | Stage | Runs |
|---|---|---|
| Indexing | Upload → Extract → Chunk → Embed → Index → Persist | once per configuration |
| Query | Embed question → Retrieve top-k → Build prompt → Generate | once per question |

### The experiment this prepares for

PT-F1 is the setup for a controlled comparison of `CHUNK_SIZE` and `CHUNK_OVERLAP`. Everything
else — the corpus, the embedding model, the test questions, the prompt, `TOP_K` — is held fixed so
that any difference in Precision@3, Recall@3 and MRR@3 is attributable to the chunking
configuration alone.

---
# Component I — Vector Indexing and Embedding

## 1. Install the required libraries

`faiss-cpu` provides the similarity-search index. `sentence-transformers` provides the embedding
model that turns chunks into vectors. `pypdf` extracts text from the PDF. `pandas` and `openpyxl`
build the Excel workbook used to inspect the generated chunks, which the exercise requires as
evidence that the vector store was built correctly.

`transformers` and `sentencepiece` were needed by the original notebook to run Flan-T5 locally.
They are retained so that the baseline model can still be loaded for comparison if required.

In [1]:
!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    sentencepiece \
    pandas \
    openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 14.7 MB/s eta 0:00:00


## 2. Import the libraries

`files` is the Colab helper for uploading and downloading. `PdfReader` reads the PDF page by page.
`SentenceTransformer` loads the embedding model. `faiss` holds the vector index, `numpy` supplies
the float32 matrices FAISS expects, and `pickle` serialises the chunk list and configuration so a
later session can reload the exact index that produced a set of results.

In [2]:
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

print("Libraries imported.")

Libraries imported.


## 3. Indexing parameters — the experiment variables

These four values are the only things that change between runs of the experiment.

| Parameter | Baseline | Role |
|---|---|---|
| `CHUNK_SIZE` | 1000 characters | how much text one retrieval unit holds |
| `CHUNK_OVERLAP` | 200 characters | how much text is repeated between neighbours |
| `TOP_K` | 3 | how many chunks the retriever returns |
| `EMBEDDING_MODEL_NAME` | `all-MiniLM-L6-v2` | maps text to a 384-dimensional vector |

`CHUNK_SIZE` and `CHUNK_OVERLAP` are counted in **characters**, not tokens. For English prose
roughly four characters make one token, so the 1000/200 baseline is about 250 tokens with 20%
overlap — inside the useful range, at the small end.

`TOP_K` is held at 3 throughout because the retrieval metrics for this exercise are Precision@3,
Recall@3 and MRR@3. Changing it would change the metric definition rather than the system.

In [3]:
# Change these values to experiment with chunking.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

## 4. Validate the parameters before anything expensive runs

Each check guards against a configuration that would fail silently or produce a misleading index.

The important one is `CHUNK_OVERLAP < CHUNK_SIZE`. The chunker advances by
`step = CHUNK_SIZE - CHUNK_OVERLAP`; if the overlap equalled or exceeded the size, the step would
be zero or negative and the loop would never terminate. Validating here means an invalid setting is
reported in a second rather than after the embedding pass.

In [4]:
if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size        :", CHUNK_SIZE)
print("Chunk overlap     :", CHUNK_OVERLAP)
print("Overlap percent   :", f"{100 * CHUNK_OVERLAP / CHUNK_SIZE:.1f}%")
print("Step size         :", CHUNK_SIZE - CHUNK_OVERLAP, "characters")
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

RAG INDEXING CONFIGURATION
Chunk size        : 1000
Chunk overlap     : 200
Overlap percent   : 20.0%
Step size         : 800 characters
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


## 5. Load the source document

The dataset is `Sagarino_PT-F1_Corpus.pdf` — 17 pages consolidating Republic Act No. 7394 (Consumer Act),
Republic Act No. 8792 (E-Commerce Act), Republic Act No. 10642 (Philippine Lemon Law), Republic Act
No. 11967 (Internet Transactions Act) and the DTI's published consumer guidance and complaint
procedure. The statutory text is verbatim; explanatory notes I had written for my own use were
removed so
that every retrievable passage is primary source material.

The cell accepts an already-present file before falling back to the upload widget, so re-running
the experiment with a different `CHUNK_SIZE` does not require re-uploading the same PDF each time.

In [5]:
print("=" * 70)
print("LOAD DOCUMENT")
print("=" * 70)

EXPECTED_PDF = "Sagarino_PT-F1_Corpus.pdf"

if os.path.exists(EXPECTED_PDF):
    # Already in the session: reuse it so repeated runs stay reproducible.
    pdf_file = EXPECTED_PDF
    print("Using the PDF already present in this session.")
else:
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise RuntimeError("No file was uploaded.")

    # Use the first uploaded PDF
    pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)
print("Size:", round(os.path.getsize(pdf_file) / 1024, 1), "KB")

LOAD DOCUMENT


Saving Sagarino_PT-F1_Corpus.pdf to Sagarino_PT-F1_Corpus.pdf

Selected file:
Sagarino_PT-F1_Corpus.pdf
Size: 50.1 KB


## 6. Extract the text, page by page

`PdfReader` returns one text block per page. Each block is whitespace-normalised, because PDF
extraction inserts line breaks wherever the layout wrapped a line, and those breaks carry no
meaning. Pages that yield no text are dropped.

The page number is captured alongside the text and carried through chunking into retrieval. That
is what lets an answer be traced back to a page of the corpus, and it is the metadata the exercise
asks us to record for each test question's evidence.

In [6]:
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))

page_lengths = [len(p["text"]) for p in pages]
print("Characters per page: min", min(page_lengths),
      "| mean", round(np.mean(page_lengths)),
      "| max", max(page_lengths))
print("Total characters extracted:", sum(page_lengths))

EXTRACTING PDF TEXT
PDF pages: 17
Pages containing text: 17
Characters per page: min 409 | mean 3580 | max 5107
Total characters extracted: 60860


## 7. Confirm the document is machine-readable

A PDF made of scanned images contains no extractable text, and every later stage would run on an
empty corpus without raising an error. This check fails loudly instead.

The exercise requires the chosen document to contain extractable text. This cell is where that
requirement is actually verified.

In [7]:
if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )

print("Text extraction verified:", len(pages), "pages are machine-readable.")

Text extraction verified: 17 pages are machine-readable.


## 8. Split the pages into chunks

This is the function the whole experiment turns on.

The loop walks each page with a sliding window: it takes `CHUNK_SIZE` characters, stores them, then
advances by `step = CHUNK_SIZE - CHUNK_OVERLAP`. Because the step is smaller than the window, the
tail of one chunk reappears at the head of the next — that repetition is the overlap, and it is
what keeps an idea readable when it would otherwise be cut in half at a boundary.

**One behaviour deserves attention.** Chunking restarts at offset 0 on every page, so a page break
is always a hard chunk boundary regardless of `CHUNK_SIZE`, and overlap never bridges two pages.
A passage split across a page break is therefore split across chunks no matter how the parameters
are set. This is a property of the provided implementation, and it is worth remembering when
interpreting the retrieval results: a question whose evidence straddles a page break is harder for
reasons that have nothing to do with the chunk size being tested.

In [8]:
def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

print("Chunks created:", len(chunks))

Chunks created: 85


## 9. Report the chunking statistics

These numbers are the first diagnostic of a configuration.

The mean will sit below `CHUNK_SIZE` because the last chunk of every page is a remainder, and with
a 17-page document there are 17 such remainders. The minimum shows the smallest of them — a very
small value means a page ended just after a boundary and produced a fragment too short to retrieve
usefully. The chunk count is what drives embedding time and index size, so it is also the cost side
of the trade-off being measured.

In [9]:
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)

print(
    "Chunks shorter than half the target:",
    sum(1 for n in chunk_lengths if n < CHUNK_SIZE / 2)
)

CHUNKING RESULTS
Number of chunks: 85
Average characters per chunk: 871.19
Shortest chunk: 37 characters
Longest chunk: 1000 characters
Chunks shorter than half the target: 12


## 10. Export the chunks to Excel for inspection

The exercise requires the generated chunks to be inspected in an `xlsx` file before the experiment
proceeds, as the evidence that a valid FAISS vector store can be built from the chosen document.

Each row records the chunk ID, the source document, the page it came from, its character count, the
two settings that produced it, and the chunk text itself. Recording the settings on every row
matters: once several configurations have been run, the workbooks are self-identifying and cannot
be confused with one another.

In [10]:
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Name the workbook after the source PDF and the configuration that produced it.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_{CHUNK_SIZE}_{CHUNK_OVERLAP}.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: Sagarino_PT-F1_Corpus_chunks_1000_200.xlsx
Chunks exported: 85
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            1             1000
        1            1              560
        2            2              999
        3            2             1000
        4            2             1000
        5            2             1000
        6            2             1000
        7            2              698
        8            3             1000
        9            3              999


## 11. Download the workbook

Kept as a separate cell from the export so that re-running the inspection does not force another
download, and so that a failure in the browser download does not look like a failure in the export.

In [11]:
# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Print sample chunks

Reading the first few chunks is the fastest way to catch a bad configuration before spending time
on embeddings. What to look for: does a chunk begin mid-sentence, does it end mid-rule, and can it
be understood without the chunk on either side of it. A chunk that cannot stand alone will retrieve
poorly no matter how good the embedding model is.

In [12]:
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])

SAMPLE CHUNKS

Chunk ID : 0
Page     : 1
Characters: 1000
----------------------------------------------------------------------
Philippine Consumer-Remedy Corpus A consolidated reference of the statutes and agency guidance that govern remedies for a defective consumer purchase in the Philippines. Compiled for ITCC508 Lab Exercise PT-F1 by Mark Emmanuel P. Sagarino. College of Computer Studies, Jose Rizal University. Contents PART I - REPUBLIC ACT NO. 7394, THE CONSUMER ACT OF THE PHILIPPINES (1992) PART II - REPUBLIC ACT NO. 8792, THE ELECTRONIC COMMERCE ACT OF 2000 PART III - REPUBLIC ACT NO. 10642, THE PHILIPPINE LEMON LAW (2014) PART IV - REPUBLIC ACT NO. 11967, THE INTERNET TRANSACTIONS ACT OF 2023 PART V - DEPARTMENT OF TRADE AND INDUSTRY: CONSUMER GUIDANCE AND COMPLAINT PROCEDURE Sources and reuse Statutory text is reproduced verbatim from the LawPhil Project of the Arellano Law Foundation and the Supreme Court E-Library, both of which publish the enacted text. Republic Acts are

## 13. Load the embedding model

`all-MiniLM-L6-v2` maps a passage to a 384-dimensional vector positioned so that passages with
similar meaning land close together. It is small and fast, which matters here because the
experiment re-embeds the whole corpus once per configuration.

This model is held constant across every run. If it changed between configurations, a difference in
retrieval scores could not be attributed to the chunking.

In [13]:
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)

LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: all-MiniLM-L6-v2


## 14. Collect the chunk texts

The embedder takes a flat list of strings, but the metadata — chunk ID and page — must not be lost.
The list below preserves order, and position `i` in it corresponds to position `i` in both the
embedding matrix and the `chunks` list. That shared ordering is what the lookup in section 17 relies
on to turn a FAISS row number back into a citable page.

In [14]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

print("Texts queued for embedding:", len(chunk_texts))

Texts queued for embedding: 85


## 15. Create the document embeddings

`normalize_embeddings=True` scales every vector to unit length. This is what makes the inner-product
index in the next section equivalent to cosine similarity, so the scores returned by a search are
bounded in a comparable range rather than being influenced by passage length.

The cast to `float32` is required: FAISS stores single-precision vectors, and passing float64 raises
an error at insertion.

In [15]:
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)

CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Embedding matrix shape:
(85, 384)


## 16. Build the FAISS vector store

`IndexFlatIP` is an exact index: a search compares the query against every stored vector rather
than approximating. With a corpus of this size that is instant, and it removes approximation as a
source of variation between configurations — a difference in results must come from the chunking.

`IP` is inner product. On the unit-length vectors produced in the previous cell, inner product is
cosine similarity.

The printed `index.ntotal` should equal the chunk count. A mismatch means vectors were added twice,
which silently halves the effective retrieval depth, so the two numbers are compared here rather
than assumed.

In [16]:
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)

print(
    "INTEGRITY CHECK   :",
    "OK - one vector per chunk"
    if index.ntotal == len(chunks)
    else f"FAILED - {index.ntotal - len(chunks)} unexpected vector(s)"
)

CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 85
INTEGRITY CHECK   : OK - one vector per chunk


## 17. Build the chunk lookup

FAISS returns row numbers, not text. This dictionary maps a row number back to the chunk that
produced it, which is how a search result recovers its page number and its text.

The mapping is correct only because the embedding matrix was built from `chunk_texts` in order and
nothing has reordered either list since.

In [17]:
# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}

print("Lookup entries:", len(chunk_lookup))

Lookup entries: 85


## 18. Define the retrieval function

This is the query-time half of the pipeline, and the step the retrieval metrics score.

The question is embedded by **the same model and with the same normalisation** used on the chunks.
This is not optional: vectors produced by different models occupy different spaces, and comparing
them returns numbers that look like similarities but mean nothing.

`search_k` is clamped to the number of stored vectors so that a small corpus does not produce
padding results with index `-1`. Each result carries its rank, chunk ID, page, similarity score and
text — everything needed to score Precision@3, Recall@3 and MRR@3 by hand.

In [18]:
def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results


print("Retrieval function ready.")

Retrieval function ready.


## 19. Smoke-test the vector store

The original notebook asked for a question through `input()`. That blocks a *Run all* and leaves no
output in a submitted notebook, so the test query is fixed here instead. It is deliberately a
question whose answer we already know the location of — Section 5 of the Lemon Law, on page 10 of
the corpus — so the printed page number is itself the test.

If this returns chunks from the right page, indexing, embedding and retrieval are all wired
correctly and Component II can be trusted to be testing the model rather than the plumbing.

In [19]:
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = "How many separate repair attempts are required under the Lemon Law?"

print("Test query:", test_query)

test_results = retrieve_documents(
    test_query,
    TOP_K
)

print("\n")
print("=" * 70)
print("RETRIEVED CHUNKS")
print("=" * 70)

for result in test_results:

    print(f"\nRESULT {result['rank']}")
    print(f"Chunk ID   : {result['chunk_id']}")
    print(f"Page       : {result['page']}")
    print(f"Similarity : {result['score']:.4f}")

    print("-" * 70)

    print(result["text"][:600])

print("\nExpected evidence page: 10 (RA 10642, Section 5)")
print("Pages actually returned:", [r["page"] for r in test_results])

TESTING VECTOR STORE
Test query: How many separate repair attempts are required under the Lemon Law?


RETRIEVED CHUNKS

RESULT 1
Chunk ID   : 45
Page       : 10
Similarity : 0.7654
----------------------------------------------------------------------
The Four-Repair Rule And The Notice Requirement Section 5. Repair Attempts. - At any time within the Lemon Law rights period, and AFTER AT LEAST FOUR (4) SEPARATE REPAIR ATTEMPTS by the same manufacturer, distributor, authorized dealer or retailer FOR THE SAME COMPLAINT, and the nonconformity issue remains unresolved, the consumer may invoke his or her rights under this Act. The repair may include replacement of parts components, or assemblies. Section 6. Notice of Availment of Lemon Law Rights. - Before availing of any remedy under this Act and subject to compliance with the provisions of Sec

RESULT 2
Chunk ID   : 47
Page       : 10
Similarity : 0.6300
----------------------------------------------------------------------
rized dealer 

## 20. Persist the index, the chunks and the configuration

Three files are written so that a configuration can be reloaded exactly as it was run.

`rag_faiss.index` is the vector store. `rag_chunks.pkl` is the chunk list with its metadata, without
which the index returns row numbers that cannot be resolved. `rag_config.pkl` records the settings
that produced both.

Lecture 6 calls this freezing a version: once evaluation begins, the documents, chunks, embeddings,
index and configuration are recorded and left alone. Without it, a result cannot be attributed to
the configuration that produced it. The filenames carry the chunk settings for the same reason.

In [20]:
FAISS_FILE = f"rag_faiss_{CHUNK_SIZE}_{CHUNK_OVERLAP}.index"
CHUNKS_FILE = f"rag_chunks_{CHUNK_SIZE}_{CHUNK_OVERLAP}.pkl"
CONFIG_FILE = f"rag_config_{CHUNK_SIZE}_{CHUNK_OVERLAP}.pkl"

faiss.write_index(
    index,
    FAISS_FILE
)

with open(CHUNKS_FILE, "wb") as f:
    pickle.dump(chunks, f)

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file,
    "chunk_count": len(chunks),
    "vector_count": int(index.ntotal)
}

with open(CONFIG_FILE, "wb") as f:
    pickle.dump(config, f)

print("FAISS index saved :", FAISS_FILE)
print("Chunks saved      :", CHUNKS_FILE)
print("Configuration     :", CONFIG_FILE)

FAISS index saved : rag_faiss_1000_200.index
Chunks saved      : rag_chunks_1000_200.pkl
Configuration     : rag_config_1000_200.pkl


## 21. Indexing summary

One block recording everything that defines this run. When several configurations have been tried,
this is the output that identifies which one a set of results belongs to.

In [21]:
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Pages indexed   :", len(pages))
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)

INDEXING COMPLETE
Source document : Sagarino_PT-F1_Corpus.pdf
Pages indexed   : 17
Number of chunks: 85
Vectors stored  : 85
Chunk size      : 1000
Chunk overlap   : 200
Embedding model : all-MiniLM-L6-v2

Saved files:
   Sagarino_PT-F1_Corpus_chunks_1000_200.xlsx
   rag_faiss_1000_200.index
   rag_chunks_1000_200.pkl
   rag_config_1000_200.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


---
# Component II — Prompt and Retrieval

## 22. What changed, and why

The provided notebook ran `google/flan-t5-base` locally. Step 3 of the exercise requires the
assigned model, **`openai/gpt-oss-20b`**, served through the Groq API. The change has three
consequences worth stating before the code:

**The 512-token squeeze disappears.** Flan-T5 accepts 512 input tokens, so the original
`create_rag_prompt` had to truncate retrieved evidence mid-chunk to fit. Truncated evidence is a
confound: a retrieval failure and a truncation failure look identical in the output. The replacement
model has a far larger context, so all three retrieved chunks are passed whole and any failure is
attributable to retrieval.

**Generation is remote.** There is no local tokenizer, so token accounting uses `tiktoken` with a
character-based fallback. The budget check is retained — not because the model is likely to overflow
here, but because the experiment will later be run at larger chunk sizes, and a silent overflow
would corrupt those results.

**Determinism must be requested.** `temperature=0` replaces the original beam search. Across a
chunking experiment the same question is asked many times, and a model that varies its wording
between runs makes it impossible to tell whether an answer changed because the chunking changed.

## 23. Install the Groq client

`groq` is the official API client. `tiktoken` provides a tokenizer compatible with the
`gpt-oss` family, used only for counting — the API does its own tokenisation.

In [22]:
!pip install -q groq tiktoken

print("Groq client installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.2 MB/s eta 0:00:00
Groq client installed.


## 24. Supply the API key without storing it

`getpass` reads the key into memory for this session only. The key is never written into a cell,
never printed, and never saved to a file, so the notebook can be submitted and shared without
exposing a credential.

If the key is ever pasted into a cell by accident, it must be treated as compromised and revoked in
the Groq console immediately — a key committed to a notebook is a key that has been published.

In [23]:
import os
from getpass import getpass
from groq import Groq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (input hidden): ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client created.")

Enter your Groq API key (input hidden): ··········
Groq client created.


## 25. Resolve the model identifier at run time

Hosted models are deprecated without notice. A hard-coded identifier that works today can return
`404 model_not_found` weeks later, with nothing in the notebook having changed — this is the hosted
equivalent of model drift, and we have hit it before on a previous exercise.

This cell asks the API which models are actually available and selects the assigned model if it is
served, falling back through a ranked list if it is not. The resolved identifier is printed so that
every set of results records the model that produced it.

In [24]:
PREFERRED_MODELS = [
    "openai/gpt-oss-20b",     # assigned model for this exercise
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]

available = sorted(m.id for m in client.models.list().data)

print("Models available on this key:", len(available))

LLM_NAME = next((m for m in PREFERRED_MODELS if m in available), None)

if LLM_NAME is None:
    raise RuntimeError(
        "None of the preferred models are available. Available models:\n"
        + "\n".join(available)
    )

print("Resolved model  :", LLM_NAME)
print("Assigned model served:", "openai/gpt-oss-20b" in available)

Models available on this key: 11
Resolved model  : openai/gpt-oss-20b
Assigned model served: True


## 26. Generation settings and token accounting

`MAX_INPUT_TOKENS` is set well above what three chunks require, so evidence is never truncated at
the baseline configuration. The budget is still enforced, because later runs of the experiment will
use larger chunks and an overflow must raise rather than pass silently.

`MAX_NEW_TOKENS` is 1024, and the reason it is that large is specific to this model.

`gpt-oss-20b` is a **reasoning model**: it produces internal reasoning tokens before it emits an
answer, and on Groq those reasoning tokens are drawn from the same completion budget. A first run of
this notebook set the budget to 320 and two questions came back with no answer at all, both having
used exactly 320 tokens — the budget was spent reasoning and nothing was left to say. The easy
question and the guardrail question, which needed 157 and 187 tokens, were unaffected.

Two settings fix it. `REASONING_EFFORT = "low"` tells the model to think briefly, and the larger
budget leaves room for the answer even when it does not. `reasoning_effort` is specific to the
`gpt-oss` family, so the call falls back gracefully if the served model does not accept it.

`RESERVED_TOKEN_MARGIN` keeps a little headroom so that a prompt sitting exactly on the limit does
not fail at the API boundary.

In [25]:
MAX_INPUT_TOKENS = 8000
MAX_NEW_TOKENS = 1024
RESERVED_TOKEN_MARGIN = 64
TEMPERATURE = 0.0
REASONING_EFFORT = "low"   # gpt-oss only; ignored by other models

try:
    import tiktoken
    _ENC = tiktoken.get_encoding("o200k_base")

    def count_tokens(text):
        return len(_ENC.encode(text))

    _COUNTER = "tiktoken o200k_base"

except Exception:
    def count_tokens(text):
        # Fallback: about four characters per token for English prose.
        return max(1, len(text) // 4)

    _COUNTER = "character estimate"

print("Token counter      :", _COUNTER)
print("Max input tokens   :", MAX_INPUT_TOKENS)
print("Max new tokens     :", MAX_NEW_TOKENS)
print("Temperature        :", TEMPERATURE)
print("Reasoning effort   :", REASONING_EFFORT)

Token counter      : tiktoken o200k_base
Max input tokens   : 8000
Max new tokens     : 1024
Temperature        : 0.0
Reasoning effort   : low


## 27. Keep the retrieval ranking

FAISS already returns results ordered by similarity. This function makes that ordering explicit and
guarantees it survives into the prompt, because position matters: a model reads the first piece of
evidence as the most relevant, and silently reordering the context would change the answer without
changing the retrieval.

In [26]:
def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )


print("Ranking function ready.")

Ranking function ready.


## 28. The grounding prompt and the guardrails

This is the deliverable the exercise asks us to report, so each rule is justified against a failure
we expect in this specific domain.

**Rule 1 — only the Evidence.** Language models are trained largely on English-language sources,
which means a great deal of American and European consumer law. The European Union's fourteen-day
right to withdraw from a purchase has no Philippine counterpart, but it is the single most likely
thing for an ungrounded model to produce here. The rule exists to make that failure impossible
rather than unlikely.

**Rule 2 — cite the provision and the page.** An answer that cannot be traced is not verifiable. The
citation also separates two failure modes that otherwise look alike: a correct remedy attributed to
the wrong statute is wrong in a way that matters, because it sends a consumer to the wrong forum on
the wrong timetable.

**Rule 3 — one canonical refusal.** A fixed string makes refusals machine-countable. Free-form
refusals vary between runs and cannot be scored.

**Rule 4 — platform policy is not law.** A marketplace's refund window is a commercial term that
appears nowhere in the corpus. Without this rule the model is liable to invent a plausible number.

**Rule 5 — quote figures exactly.** Thirty days, twelve months, twenty thousand kilometres, four
repair attempts, seven calendar days. These are the answers; an approximation is a wrong answer.

**Rule 6 — state exclusions first.** The hardest questions in this domain are near-misses, where a
rule looks applicable and is not. Requiring the model to name the excluding provision before giving
the rule that does apply is what converts a silent mistake into a visible, checkable step.

In [27]:
REFUSAL = "I cannot find the answer in the provided document."
NOT_ESTABLISHED = "The evidence does not establish whether this item is covered."

SYSTEM_PROMPT = (
    "You are a retrieval-grounded assistant for Philippine consumer-remedy law. "
    "You answer only from the Evidence supplied in the user message.\n"
    "\n"
    "ANSWER FORMAT - use exactly this shape, including the Scope line:\n"
    "  <one or two sentences that directly answer the question>\n"
    "  Supporting evidence: \"<verbatim quote from the Evidence>\" [Page N]\n"
    "  Scope: <the Evidence provision that brings the item named in the question "
    "within the rule you applied, quoted and cited - or the words 'not established "
    "in the Evidence'>\n"
    "\n"
    "Rules:\n"
    "1. Use only the Evidence. Do not use prior knowledge, and do not import rules "
    "from the consumer law of any other country.\n"
    "2. Read every Evidence block before answering. They may come from different "
    "statutes; use the one that addresses the question asked.\n"
    "3. Always give the answer sentence first. A quotation on its own is not an "
    "answer. Then give the supporting quote, reproduced word for word from the "
    "Evidence, with the page cited immediately after it. Never cite a page you did "
    "not quote from.\n"
    "4. Scope check. Before stating that a rule applies to the item named in the "
    "question, confirm that some Evidence block contains the definition or coverage "
    "provision that brings that item within the rule. If no block defines or lists the "
    "item, you must not assume it is covered: answer " + NOT_ESTABLISHED + " and name "
    "the definition or coverage provision that is missing. A broad category in the "
    "Evidence does not automatically include a narrower item named in the question. "
    "If the question names a specific kind of thing and the Evidence only uses a "
    "wider category, you must find the provision that defines that category before "
    "treating the item as falling inside it.\n"
    "5. If the Evidence shows that a rule does not cover the item, say so first, quote "
    "and cite the provision that excludes it, then give the provision that does apply.\n"
    "6. Refuse only as a last resort, and only after checking every Evidence block. "
    "If, having read them all, none contains the answer, reply with exactly this "
    "sentence and nothing else: " + REFUSAL + " Do not refuse merely because you are "
    "unsure which block is the right one, or because the question uses different "
    "wording from the Evidence.\n"
    "7. A platform's own refund window, a seller's terms, or any commercial policy is "
    "not law. If the question asks for one, apply rule 6.\n"
    "8. Quote figures exactly as they appear: days, months, kilometres, number of "
    "repair attempts, amounts.\n"
    "\n"
    "Be concise. Do not restate the question."
)


def create_rag_prompt(question, retrieved_documents):
    """
    Build the user message: the question, then the retrieved evidence,
    each block labelled with its page and chunk so the model can cite it.

    Evidence is included whole. If the running total would exceed the input
    budget, the remaining chunks are dropped rather than cut mid-passage,
    because a half-chunk is indistinguishable in the output from a retrieval
    miss.
    """

    ranked_documents = prioritize_documents(question, retrieved_documents)

    header = f"Question: {question}\n\nEvidence:\n"
    footer = "\n\nAnswer:"

    budget = (
        MAX_INPUT_TOKENS
        - count_tokens(SYSTEM_PROMPT)
        - count_tokens(header)
        - count_tokens(footer)
        - MAX_NEW_TOKENS
        - RESERVED_TOKEN_MARGIN
    )

    if budget <= 0:
        raise ValueError(
            "The system prompt and question leave no room for evidence."
        )

    selected, dropped, used = [], 0, 0

    for result in ranked_documents:

        block = (
            f"[Page {result['page']}, Chunk {result['chunk_id']}, "
            f"similarity {result['score']:.4f}]\n{result['text']}"
        )

        cost = count_tokens(block) + 2

        if used + cost > budget:
            dropped += 1
            continue

        selected.append(block)
        used += cost

    if not selected:
        raise ValueError(
            "No retrieved chunk fits the input budget. Reduce CHUNK_SIZE "
            "or raise MAX_INPUT_TOKENS."
        )

    prompt = header + "\n\n".join(selected) + footer

    total = count_tokens(SYSTEM_PROMPT) + count_tokens(prompt)

    if total > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {total} tokens, exceeding the "
            f"{MAX_INPUT_TOKENS}-token limit."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Question            :", question)
    print("Chunks retrieved    :", len(retrieved_documents))
    print("Chunks included     :", len(selected))
    print("Chunks dropped      :", dropped)
    print("Pages in evidence   :", sorted({r["page"] for r in ranked_documents}))
    print("Prompt tokens       :", total, "of", MAX_INPUT_TOKENS)

    return prompt


print("Prompt builder ready.")
print("Refusal string:", REFUSAL)

Prompt builder ready.
Refusal string: I cannot find the answer in the provided document.


## 29. Generate the answer

The system prompt and the user message are sent as separate roles, which is how a chat model is
meant to receive an instruction it must not treat as data.

`temperature=0` makes the call deterministic, so repeating a question returns the same answer and a
difference between runs can be attributed to the configuration rather than to sampling.

Three distinct outcomes must not be confused with one another, because they mean different things
and would be scored differently: an **API error** (the request never completed), a **truncation**
(the completion budget ran out before an answer was emitted), and a **refusal** (the model answered,
and its answer was that the evidence does not support one). The function below reports
`finish_reason` so the second case names itself instead of appearing as an empty answer.

In [28]:
def generate_answer(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """
    Send the grounded prompt to the Groq-hosted model and return the answer.

    Reports finish_reason so that a truncated completion is identified as a
    truncation rather than being mistaken for an empty or refused answer.
    """

    kwargs = dict(
        model=LLM_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=TEMPERATURE,
        max_completion_tokens=max_new_tokens,
    )

    # reasoning_effort is specific to the gpt-oss family.
    if REASONING_EFFORT and "gpt-oss" in LLM_NAME:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    try:
        response = client.chat.completions.create(**kwargs)

    except Exception as exc:
        if "reasoning_effort" in str(exc):
            kwargs.pop("reasoning_effort", None)
            try:
                response = client.chat.completions.create(**kwargs)
            except Exception as exc2:
                return f"API ERROR: {exc2}"
        else:
            return f"API ERROR: {exc}"

    choice = response.choices[0]
    answer = (choice.message.content or "").strip()
    finish = getattr(choice, "finish_reason", None)

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Model          :", LLM_NAME)
    print("Temperature    :", TEMPERATURE)
    print("Reasoning      :", kwargs.get("reasoning_effort", "not requested"))
    print("Finish reason  :", finish)
    try:
        print("Prompt tokens  :", response.usage.prompt_tokens)
        print("Output tokens  :", response.usage.completion_tokens, "of", max_new_tokens)
    except Exception:
        pass

    if not answer:

        if finish == "length":
            return (
                "TRUNCATED: the completion budget was exhausted before an answer "
                "was emitted. Raise MAX_NEW_TOKENS or lower REASONING_EFFORT."
            )

        # Some reasoning models place text in a separate field when content is empty.
        trailing = (getattr(choice.message, "reasoning", None) or "").strip()
        if trailing:
            return trailing

        return (
            "The model returned no content and no truncation signal. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer


print("Generator ready.")

Generator ready.


## 30. Check that the answer is grounded

A guardrail that is never verified is a hope, not a control. This function performs two cheap,
mechanical checks on every answer.

The first reads back the `[Page N]` citations and asks whether those pages were actually among the
chunks retrieved.

**That test alone proved too weak.** On an earlier run the model answered a question about the
Internet Transactions Act using a chunk from the Lemon Law, invented a procedure that appears
nowhere in the statute, and cited a page that *had* been retrieved. The page check passed it. The
answer was wrong.

So the second check verifies the **quoted text**. The prompt now requires the model to reproduce
verbatim the clause it relies on, and this function confirms that each quoted passage actually
occurs in the retrieved chunks. A quote either appears in the evidence or it does not, which is a
far harder thing to fake than a page number.

The third detects the canonical refusal and the "not established" response, so cautious answers are
counted separately from wrong ones rather than being scored alike.

**What this check cannot do.** Grounded is not the same as correct. A quotation can be perfectly
genuine and the conclusion drawn from it still wrong, if the passage that would have changed the
answer was never retrieved. That failure is invisible here and is caught instead by the evidence
coverage reported in the test-set runner: an answer built on one of the three passages a question
needs is not a safe answer, however honestly it quotes that one.

In [29]:
import re as _re


def _norm(t):
    """Whitespace- and quote-insensitive form, for substring matching."""
    t = t.replace("’", "'").replace("‘", "'")
    t = t.replace("“", '"').replace("”", '"')
    t = t.replace("‑", "-").replace("–", "-").replace("—", "-")
    return _re.sub(r"\s+", " ", t).strip().lower()


def check_grounding(answer, retrieved_documents):
    """
    Verify three things, in increasing order of strictness:

      pages_ok  - every [Page N] cited was actually retrieved
      quotes_ok - every quoted passage appears verbatim in the retrieved text
      grounded  - the answer quoted at least one passage and all of them check out

    The page test alone is too weak. An answer can cite a page that was
    retrieved while making a claim that page does not support, which is exactly
    how a confident wrong answer passes an inspection that only counts pages.
    Checking the quotes closes that gap, because a quote either appears in the
    retrieved text or it does not.
    """

    stripped = answer.strip()
    refused = _norm(stripped).rstrip(".") == _norm(REFUSAL).rstrip(".")

    # Declining is detected by pattern rather than by exact wording. An earlier
    # version matched the canonical sentence only, so a model that declined
    # correctly but phrased it in its own words was scored as if it had
    # answered - penalising exactly the behaviour the guardrail asks for.
    _DECLINE = (
        "does not establish", "do not establish", "does not specify",
        "do not specify", "is not stated", "are not stated",
        "cannot be determined from", "not contained in the provided",
        "not found in the provided", "does not contain",
    )
    _n = _norm(stripped)
    not_established = any(d in _n for d in _DECLINE)

    cited = {int(n) for n in _re.findall(r"\[Page\s+(\d+)", answer)}
    retrieved = {r["page"] for r in retrieved_documents}
    unsupported = sorted(cited - retrieved)

    corpus = _norm(" ".join(r["text"] for r in retrieved_documents))

    # Normalise the quotation marks BEFORE extracting. Models mix straight
    # quotes with typographic ones, and an earlier version of this function
    # searched only for straight quotes - so a correctly quoted answer that
    # happened to use curly quotes was reported as having quoted nothing.
    flat = (answer
            .replace("“", '"').replace("”", '"')
            .replace("«", '"').replace("»", '"'))

    # Quoted spans long enough to be a real citation rather than a stray word.
    quotes = _re.findall(r'"([^"]{25,})"', flat)

    def _quote_ok(q):
        """
        A quote may be abridged with an ellipsis. Treat it as verified when
        every fragment either side of the ellipsis appears in the evidence,
        so that shortening a long passage is not mistaken for inventing one.
        """
        if _norm(q) in corpus:
            return True
        parts = [p for p in _re.split(r"\u2026|\.\.\.", q) if len(p.strip()) >= 15]
        return bool(parts) and all(_norm(p) in corpus for p in parts)

    bad_quotes = [q for q in quotes if not _quote_ok(q)]

    pages_ok = bool(cited) and not unsupported
    quotes_ok = bool(quotes) and not bad_quotes

    return {
        "refused": refused,
        "not_established": not_established,
        "cited_pages": sorted(cited),
        "retrieved_pages": sorted(retrieved),
        "unsupported_citations": unsupported,
        "quotes_found": len(quotes),
        "quotes_not_in_evidence": bad_quotes,
        "pages_ok": pages_ok,
        "quotes_ok": quotes_ok,
        # Declining is grounded behaviour, not a hallucination. Both forms of
        # declining - the flat refusal and the narrower "not established" -
        # are the system correctly reporting the limits of its evidence, and
        # penalising them would reward a confident guess over an honest one.
        "grounded": refused or not_established or (pages_ok and quotes_ok),
    }


print("Grounding check ready.")
print("Declining counts as grounded. Any other answer must quote and cite.")

Grounding check ready.
Declining counts as grounded. Any other answer must quote and cite.


## 31. The complete RAG query function

One call that runs the query-time half of the pipeline end to end: retrieve, build the prompt,
generate, then check the grounding.

The empty-retrieval branch returns the canonical refusal rather than calling the model at all. With
no evidence there is nothing to ground an answer in, and asking anyway would invite the model to
answer from memory.

In [30]:
def rag_query(question, top_k=TOP_K):
    """
    Retrieve evidence, build a grounded prompt, generate an answer,
    and verify that the answer cites what was actually retrieved.

    Returns (answer, retrieved_documents, prompt, grounding_report).
    """

    question = question.strip()

    if not question:
        return "Please enter a valid question.", [], "", None

    retrieved_documents = retrieve_documents(question, top_k=top_k)

    if not retrieved_documents:
        return REFUSAL, [], "", None

    prompt = create_rag_prompt(question, retrieved_documents)

    answer = generate_answer(prompt)

    report = check_grounding(answer, retrieved_documents)

    return answer.strip(), retrieved_documents, prompt, report


print("RAG query function ready.")

RAG query function ready.


## 32. The test set

Three questions are required, one at each difficulty level, together with the expected answer and
the exact passage that supports it. Difficulty here comes from the amount of evidence the answer
needs, not from vague wording — every question below is fully answerable from the corpus.

| ID | Level | Why it sits at that level |
|---|---|---|
| Q1 | Easy | One explicit figure in one passage on one page. |
| Q2 | Average | Two related facts from the same section: the required step, and when it is complete. |
| Q3 | Difficult | Three passages across two statutes, and an exclusion that must be recognised before the applicable rule can be given. |

A fourth item is included as a **guardrail check**. It is not one of the three graded questions: it
is deliberately unanswerable from the corpus, and its purpose is to confirm that the refusal rule
fires instead of the model inventing a plausible number. Lecture 6 calls this an in-scope,
unanswerable question.

In [31]:
TEST_QUESTIONS = [
    {
        "id": "Q1",
        "difficulty": "Easy",
        "question": (
            "Under the Philippine Lemon Law, how many separate repair attempts "
            "for the same complaint must fail before a consumer may invoke their "
            "rights under the Act?"
        ),
        "expected_answer": "Four (4) separate repair attempts.",
        "evidence": "Sagarino_PT-F1_Corpus.pdf, page 10 - RA 10642, Section 5 (Repair Attempts).",
        "evidence_pages": [10],
        "answerable": True,
    },
    {
        "id": "Q2",
        "difficulty": "Average",
        "question": (
            "Under the Internet Transactions Act, what must an aggrieved online "
            "buyer do before filing a complaint before any court or government "
            "agency, and when is that step deemed exhausted?"
        ),
        "expected_answer": (
            "The buyer must first use the internal redress mechanism of the digital "
            "platform, e-marketplace or e-retailer. It is deemed exhausted if the "
            "complaint remains unresolved after seven (7) calendar days from filing."
        ),
        "evidence": "Sagarino_PT-F1_Corpus.pdf, page 14 - RA 11967, Section 24 (Internal Redress Mechanism).",
        "evidence_pages": [14],
        "answerable": True,
    },
    {
        "id": "Q3",
        "difficulty": "Difficult",
        "question": (
            "A consumer's brand-new motorcycle has failed four separate repair "
            "attempts within six months of delivery. Can they invoke the Philippine "
            "Lemon Law? If not, which statute governs and what remedy does it give?"
        ),
        "expected_answer": (
            "No. Section 3 (j) of RA 10642 defines a motor vehicle as a four-wheeled "
            "road vehicle and expressly excludes motorcycles, and Section 4 limits "
            "coverage to brand new motor vehicles, so the four-repair rule in Section 5 "
            "does not apply. Article 100 of RA 7394 governs instead: if the imperfection "
            "is not corrected within thirty (30) days, the consumer may demand "
            "replacement of the product, reimbursement of the amount paid, or a "
            "proportionate price reduction."
        ),
        "evidence": (
            "Sagarino_PT-F1_Corpus.pdf, page 9 - RA 10642, Section 3 (j) and Section 4; "
            "page 10 - RA 10642, Section 5; page 4 - RA 7394, Article 100."
        ),
        "evidence_pages": [9, 10, 4],
        "answerable": True,
    },
    {
        "id": "G1",
        "difficulty": "Guardrail (not graded)",
        "question": "How many days does Shopee give a buyer to request a refund?",
        "expected_answer": REFUSAL,
        "evidence": (
            "Not in the corpus. A marketplace refund window is a commercial term set "
            "by contract, not a statutory entitlement, and appears in none of the "
            "five sources."
        ),
        "evidence_pages": [],
        "answerable": False,
    },
]

print("Test questions loaded:", len(TEST_QUESTIONS))
for q in TEST_QUESTIONS:
    print(f"  {q['id']} [{q['difficulty']}] {q['question'][:62]}...")

Test questions loaded: 4
  Q1 [Easy] Under the Philippine Lemon Law, how many separate repair attem...
  Q2 [Average] Under the Internet Transactions Act, what must an aggrieved on...
  Q3 [Difficult] A consumer's brand-new motorcycle has failed four separate rep...
  G1 [Guardrail (not graded)] How many days does Shopee give a buyer to request a refund?...


## 33. Run the test set without interaction

The original notebook ended in an `input()` loop. That blocks a *Run all*, and a notebook submitted
after a *Run all* would show no answers at all — the outputs a grader needs would be missing.

This cell runs every test question in order and prints the answer, the retrieved pages, and whether
those pages match the evidence recorded for the question. The interactive loop is kept, but moved to
the end and disabled by default.

In [32]:
results_log = []

for item in TEST_QUESTIONS:

    print("\n" + "=" * 70)
    print(f"{item['id']}  |  {item['difficulty']}")
    print("=" * 70)
    print("Question :", item["question"])

    answer, sources, prompt, report = rag_query(item["question"], top_k=TOP_K)

    retrieved_pages = [s["page"] for s in sources]
    expected_pages = item["evidence_pages"]

    # Coverage, not a single hit. A question needing three passages is not
    # satisfied by retrieving one of them, and an any-match test would hide
    # exactly the failure this experiment exists to find.
    if not item["answerable"]:
        found, coverage, full = [], "n/a (unanswerable)", None
    else:
        found = sorted(set(expected_pages) & set(retrieved_pages))
        coverage = f"{len(found)}/{len(expected_pages)}"
        full = len(found) == len(expected_pages)

    print("\n--- ANSWER ---")
    print(answer)

    print("\n--- EVIDENCE CHECK ---")
    print("Expected page(s) :", expected_pages or "none")
    print("Retrieved page(s):", retrieved_pages)
    print("Evidence coverage:", coverage,
          f"- found {found}" if item["answerable"] else "")
    if item["answerable"] and len(found) < len(expected_pages):
        missing = sorted(set(expected_pages) - set(retrieved_pages))
        print("MISSING evidence :", missing,
              "- the answer cannot be fully grounded at this configuration")
    if report:
        print("Refused          :", report["refused"])
        print("Not established  :", report["not_established"])
        print("Pages cited      :", report["cited_pages"] or "none")
        print("Unsupported cites:", report["unsupported_citations"] or "none")
        print("Quotes found     :", report["quotes_found"])
        if report["quotes_not_in_evidence"]:
            print("QUOTE NOT IN TEXT:", report["quotes_not_in_evidence"])
        print("Pages OK / Quotes OK:", report["pages_ok"], "/", report["quotes_ok"])
        print("Grounded         :", report["grounded"])

        # Grounded is not the same as safe. An answer can quote the evidence
        # it was given, faithfully, and still be wrong because the passage
        # that would have changed it was never retrieved. The verdict below
        # combines the two: an answer is safe only if it declined, or if it
        # was grounded on the complete set of passages the question needs.
        declined = report["refused"] or report["not_established"]
        if full is None:
            verdict = "SAFE - declined" if declined else "UNSAFE - answered an unanswerable item"
        elif declined:
            verdict = "SAFE - declined rather than guess"
        elif full and report["grounded"]:
            verdict = "SAFE - grounded on complete evidence"
        elif not full:
            verdict = "UNSAFE - confident answer on PARTIAL evidence"
        else:
            verdict = "UNSAFE - not grounded"
        print("VERDICT          :", verdict)

    results_log.append({
        "ID": item["id"],
        "Difficulty": item["difficulty"],
        "Evidence_Coverage": coverage,
        "Refused": report["refused"] if report else None,
        "Quotes": report["quotes_found"] if report else None,
        "Quotes_OK": report["quotes_ok"] if report else None,
        "Grounded": report["grounded"] if report else None,
        "Verdict": verdict if report else None,
        "Retrieved_Pages": retrieved_pages,
        "Answer": answer,
    })

print("\n\nTest set complete:", len(results_log), "items.")


Q1  |  Easy
Question : Under the Philippine Lemon Law, how many separate repair attempts for the same complaint must fail before a consumer may invoke their rights under the Act?

--- PROMPT DIAGNOSTICS ---
Question            : Under the Philippine Lemon Law, how many separate repair attempts for the same complaint must fail before a consumer may invoke their rights under the Act?
Chunks retrieved    : 3
Chunks included     : 3
Chunks dropped      : 0
Pages in evidence   : [1, 10]
Prompt tokens       : 1259 of 8000

--- GENERATION DIAGNOSTICS ---
Model          : openai/gpt-oss-20b
Temperature    : 0.0
Reasoning      : low
Finish reason  : stop
Prompt tokens  : 1333
Output tokens  : 181 of 1024

--- ANSWER ---
The consumer may invoke their rights after **four (4) separate repair attempts** for the same complaint have failed.  
Supporting evidence: "AFTER AT LEAST FOUR (4) SEPARATE REPAIR ATTEMPTS by the same manufacturer, distributor, authorized dealer or retailer FOR THE SAME COMPLA

## 34. Results summary

A single table for this configuration. It is not yet the experiment — Precision@3, Recall@3 and
MRR@3 come next — but it is the record that shows the pipeline, the prompt and the guardrails all
behave as intended at the 1000/200 baseline before any parameter is varied.

The last column is the one to read. **Grounded is not the same as safe.**

An answer can quote the evidence it was given, faithfully and verifiably, and still be wrong,
because the passage that would have changed it was never retrieved. Such an answer passes every
check that looks only at the model, and it is the most dangerous output the system can produce: it
is confident, it is citable, and it is incorrect.

The verdict therefore combines two separate measurements. An answer is **safe** only if it declined,
or if it was grounded on the *complete* set of passages the question needs. An answer that is
grounded on partial evidence and did not decline is marked **UNSAFE**, no matter how well it quoted
what it had.

What should be true of a configuration worth taking forward: every graded question safe, and G1
declining.

In [33]:
summary = pd.DataFrame(results_log)[
    ["ID", "Difficulty", "Evidence_Coverage", "Quotes_OK", "Grounded", "Verdict"]
]

print("=" * 78)
print(f"BASELINE RESULTS  |  chunk {CHUNK_SIZE} / overlap {CHUNK_OVERLAP} / top-k {TOP_K}")
print(f"Model: {LLM_NAME}   Embeddings: {EMBEDDING_MODEL_NAME}   Chunks: {len(chunks)}")
print("=" * 78)
print(summary.to_string(index=False))

summary.to_csv(f"ptf1_baseline_{CHUNK_SIZE}_{CHUNK_OVERLAP}.csv", index=False)
print("\nSaved:", f"ptf1_baseline_{CHUNK_SIZE}_{CHUNK_OVERLAP}.csv")

BASELINE RESULTS  |  chunk 1000 / overlap 200 / top-k 3
Model: openai/gpt-oss-20b   Embeddings: all-MiniLM-L6-v2   Chunks: 85
ID             Difficulty  Evidence_Coverage  Quotes_OK  Grounded                                       Verdict
Q1                   Easy                1/1       True      True          SAFE - grounded on complete evidence
Q2                Average                1/1       True      True          SAFE - grounded on complete evidence
Q3              Difficult                1/3       True      True UNSAFE - confident answer on PARTIAL evidence
G1 Guardrail (not graded) n/a (unanswerable)      False      True                               SAFE - declined

Saved: ptf1_baseline_1000_200.csv


## 35. Retrieval diagnostic — where the missing evidence actually ranks

If a question's evidence coverage is below full, two very different explanations are possible, and
the summary table cannot tell them apart.

Either the evidence sits **just outside** the top 3 — in which case the chunking is close to
adequate and a different size or overlap may be enough to lift it in — or it is **far down the
ranking**, which means the chunk containing it does not resemble the question at all and no
adjustment to the boundary will help.

This cell searches deeper than the scored depth and reports the rank at which each expected page
first appears. `TOP_K` is unchanged: `DIAG_K` is used only here, for diagnosis, and never for
scoring. Precision@3, Recall@3 and MRR@3 remain defined at 3.

This is the measurement that motivates the chunk-size experiment, so it belongs in the initial
preparation rather than being discovered later.

In [34]:
DIAG_K = 15   # diagnostic depth only - the scored metrics stay at TOP_K = 3

print("=" * 78)
print(f"RETRIEVAL DIAGNOSTIC  |  chunk {CHUNK_SIZE} / overlap {CHUNK_OVERLAP}")
print(f"Scored depth TOP_K = {TOP_K}; searching to depth {DIAG_K} for diagnosis only")
print("=" * 78)

diag_rows = []

for item in TEST_QUESTIONS:

    if not item["answerable"]:
        continue

    deep = retrieve_documents(item["question"], top_k=DIAG_K)

    print(f"\n{item['id']}  [{item['difficulty']}]")
    print("  Pages by rank:", [(r["rank"], r["page"]) for r in deep[:DIAG_K]])

    for page in item["evidence_pages"]:

        ranks = [r["rank"] for r in deep if r["page"] == page]
        first = ranks[0] if ranks else None

        if first is None:
            verdict = f"not in top {DIAG_K} - the chunk does not resemble the question"
        elif first <= TOP_K:
            verdict = "retrieved at the scored depth"
        else:
            verdict = f"missed at k={TOP_K}, would be found at k={first}"

        print(f"    page {page:>2}: first rank {str(first):>4}  -  {verdict}")

        diag_rows.append({
            "ID": item["id"],
            "Expected_Page": page,
            "First_Rank": first,
            "Within_TopK": bool(first and first <= TOP_K),
        })

diag = pd.DataFrame(diag_rows)

print("\n" + "=" * 78)
print("SUMMARY")
print("=" * 78)
print(diag.to_string(index=False))

within = int(diag["Within_TopK"].sum())
total = len(diag)
print(f"\nEvidence passages inside the scored depth: {within} of {total}")
print("Any passage outside it is a retrieval failure, not a generation failure -")
print("no prompt change can repair it, which is what the chunking experiment is for.")

diag.to_csv(f"ptf1_retrieval_diag_{CHUNK_SIZE}_{CHUNK_OVERLAP}.csv", index=False)
print("\nSaved:", f"ptf1_retrieval_diag_{CHUNK_SIZE}_{CHUNK_OVERLAP}.csv")

RETRIEVAL DIAGNOSTIC  |  chunk 1000 / overlap 200
Scored depth TOP_K = 3; searching to depth 15 for diagnosis only

Q1  [Easy]
  Pages by rank: [(1, 10), (2, 10), (3, 1), (4, 10), (5, 10), (6, 9), (7, 3), (8, 3), (9, 4), (10, 3), (11, 14), (12, 3), (13, 16), (14, 9), (15, 12)]
    page 10: first rank    1  -  retrieved at the scored depth

Q2  [Average]
  Pages by rank: [(1, 14), (2, 14), (3, 13), (4, 14), (5, 13), (6, 16), (7, 14), (8, 16), (9, 12), (10, 13), (11, 16), (12, 12), (13, 14), (14, 12), (15, 12)]
    page 14: first rank    1  -  retrieved at the scored depth

Q3  [Difficult]
  Pages by rank: [(1, 10), (2, 10), (3, 10), (4, 9), (5, 10), (6, 3), (7, 1), (8, 9), (9, 9), (10, 3), (11, 4), (12, 11), (13, 9), (14, 11), (15, 3)]
    page  9: first rank    4  -  missed at k=3, would be found at k=4
    page 10: first rank    1  -  retrieved at the scored depth
    page  4: first rank   11  -  missed at k=3, would be found at k=11

SUMMARY
ID  Expected_Page  First_Rank  Within_TopK

## 36. Optional interactive session

The original continuous question-and-answer loop, preserved and given a name — **KonsyumerBot**.

It is **disabled by default**, and that is deliberate. `input()` blocks a *Run all*: the notebook
stops at this cell and waits, and every cell below it is left unexecuted. A notebook submitted in
that state shows no results at all.

To demonstrate it, run a clean *Run all* first, then set `RUN_INTERACTIVE = True` and run **this
cell only**. The outputs captured above are preserved, and the conversation is recorded here.

Four commands are available. `sources` reprints the evidence from the last query, `prompt` reprints
the exact text sent to the model, and `diag` prints the full grounding report — whether the answer
quoted the evidence, whether those quotes were real, and which pages it cited. Together they are
the fastest way to tell a retrieval failure from a generation failure: if `sources` shows the right
passage and the answer still went wrong, the fault is in generation.

In [35]:
BOT_NAME = "KonsyumerBot"

RUN_INTERACTIVE = False   # set True to ask your own questions

if RUN_INTERACTIVE:

    print("=" * 70)
    print(f"CONTINUOUS RAG QUESTION ANSWERING - {BOT_NAME}")
    print("=" * 70)
    print("Ask about remedies for a defective purchase under Philippine law.")
    print("Corpus: RA 7394, RA 8792, RA 10642, RA 11967 and DTI guidance.")
    print("Commands: 'exit', 'sources', 'prompt', 'diag'.")

    last_sources, last_prompt, last_report = [], None, None

    while True:

        user_question = input("\nQuestion: ").strip()

        if user_question.lower() == "exit":
            print(f"\n{BOT_NAME} session terminated.")
            break

        if not user_question:
            print("Please enter a question.")
            continue

        if user_question.lower() == "sources":
            if not last_sources:
                print("No previous query sources are available.")
                continue
            for result in last_sources:
                print(
                    f"\nRank {result['rank']} | Page {result['page']} | "
                    f"Chunk {result['chunk_id']} | Similarity {result['score']:.4f}"
                )
                print("-" * 70)
                print(result["text"])
            continue

        if user_question.lower() == "prompt":
            if last_prompt is None:
                print("No previous query prompt is available.")
                continue
            print(last_prompt)
            continue

        if user_question.lower() == "diag":
            if last_report is None:
                print("No previous query to diagnose.")
                continue
            print("\n" + "=" * 70)
            print("GROUNDING DIAGNOSTIC FOR THE LAST ANSWER")
            print("=" * 70)
            print("Refused            :", last_report["refused"])
            print("Not established    :", last_report["not_established"])
            print("Pages retrieved    :", last_report["retrieved_pages"])
            print("Pages cited        :", last_report["cited_pages"] or "none")
            print("Unsupported cites  :", last_report["unsupported_citations"] or "none")
            print("Quotes found       :", last_report["quotes_found"])
            print("Quotes not in text :", last_report["quotes_not_in_evidence"] or "none")
            print("Pages OK           :", last_report["pages_ok"])
            print("Quotes OK          :", last_report["quotes_ok"])
            print("GROUNDED           :", last_report["grounded"])
            print("\nNote: grounded means the answer quoted and cited the retrieved")
            print("evidence. It does not mean the retrieved evidence was sufficient.")
            continue

        try:
            answer, sources, prompt, report = rag_query(user_question, top_k=TOP_K)
            last_sources, last_prompt, last_report = sources, prompt, report

            print("\n" + "=" * 70)
            print(f"{BOT_NAME} ANSWER")
            print("=" * 70)
            print(answer)

            print("\nRetrieved pages:", [s["page"] for s in sources])
            if report:
                print("Grounded:", report["grounded"],
                      "| Refused:", report["refused"],
                      "| Quotes verified:", report["quotes_ok"])
            print("Type 'diag' for the full grounding report, 'sources' for the evidence.")

        except Exception as e:
            print("\nERROR:", e)

else:
    print(f"{BOT_NAME} interactive loop is disabled.")
    print("Set RUN_INTERACTIVE = True and run THIS CELL ONLY to try it,")
    print("so that the outputs already captured above are preserved.")

KonsyumerBot interactive loop is disabled.
Set RUN_INTERACTIVE = True and run THIS CELL ONLY to try it,
so that the outputs already captured above are preserved.


## 37. Submission record

The two values the exercise asks us to report, printed from the live objects rather than copied by
hand, so the submitted sheet cannot drift from what the notebook actually ran.

In [36]:
print("=" * 78)
print("PT-F1 SUBMISSION RECORD  -  Mark Emmanuel P. Sagarino")
print("=" * 78)

print("\nFINAL LANGUAGE MODEL")
print(" ", LLM_NAME, "(served through the Groq API)")

print("\nDATASET")
print(" ", pdf_file, "|", len(pages), "pages |", len(chunks), "chunks")

print("\nFIXED CONFIGURATION")
print("  Embedding model :", EMBEDDING_MODEL_NAME)
print("  Index           : FAISS IndexFlatIP on normalised vectors (cosine)")
print("  Top-K           :", TOP_K)
print("  Temperature     :", TEMPERATURE)
print("  Max new tokens  :", MAX_NEW_TOKENS)
print("  Reasoning effort:", REASONING_EFFORT)

print("\nBASELINE CHUNKING")
print("  Chunk size      :", CHUNK_SIZE, "characters")
print("  Chunk overlap   :", CHUNK_OVERLAP, "characters",
      f"({100 * CHUNK_OVERLAP / CHUNK_SIZE:.0f}%)")

print("\nFINAL PROMPT - SYSTEM MESSAGE")
print("-" * 78)
print(SYSTEM_PROMPT)

print("\nFINAL PROMPT - USER MESSAGE TEMPLATE")
print("-" * 78)
print("Question: {question}\n\nEvidence:\n"
      "[Page {page}, Chunk {chunk_id}, similarity {score}]\n{chunk_text}\n"
      "... one block per retrieved chunk ...\n\nAnswer:")
print("=" * 78)

PT-F1 SUBMISSION RECORD  -  Mark Emmanuel P. Sagarino

FINAL LANGUAGE MODEL
  openai/gpt-oss-20b (served through the Groq API)

DATASET
  Sagarino_PT-F1_Corpus.pdf | 17 pages | 85 chunks

FIXED CONFIGURATION
  Embedding model : all-MiniLM-L6-v2
  Index           : FAISS IndexFlatIP on normalised vectors (cosine)
  Top-K           : 3
  Temperature     : 0.0
  Max new tokens  : 1024
  Reasoning effort: low

BASELINE CHUNKING
  Chunk size      : 1000 characters
  Chunk overlap   : 200 characters (20%)

FINAL PROMPT - SYSTEM MESSAGE
------------------------------------------------------------------------------
You are a retrieval-grounded assistant for Philippine consumer-remedy law. You answer only from the Evidence supplied in the user message.

ANSWER FORMAT - use exactly this shape, including the Scope line:
  <one or two sentences that directly answer the question>
  Supporting evidence: "<verbatim quote from the Evidence>" [Page N]
  Scope: <the Evidence provision that brings the it